# OsteoClassAI v2 · 04 Publicar la demo (en el navegador, gratis)

Exporta el modelo de Drive a ONNX, lo cuantiza a int8 (~28 MB), verifica que dé las mismas predicciones que el original y publica `v2/web/` en un **Space estático** de Hugging Face (gratis). El modelo corre en el navegador del visitante.

Necesita un token de Hugging Face tipo **Write** (se pega solo en el recuadro de `notebook_login()`).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/OsteoClassAI/runs/convnext_tiny'
!ls -la "$RUN_DIR"

In [ ]:
!git clone -q https://github.com/walterandres23/osteoclassai.git
%cd osteoclassai/v2
!pip -q install -e . timm kagglehub onnx onnxruntime onnxscript "huggingface_hub<2.0"

## Exportar, cuantizar y comparar con el modelo original

In [ ]:
import kagglehub, pathlib
path = pathlib.Path(kagglehub.dataset_download('shashwatwork/knee-osteoarthritis-dataset-with-severity'))
root = next(p.parent for p in path.rglob('train') if p.is_dir())
!python -m osteoclass.export_onnx --ckpt "$RUN_DIR/best.pt" --out web/model.onnx --quantize --check-root "$root" --check-pred "$RUN_DIR/pred_test.csv"
!cp web/model_int8.onnx "$RUN_DIR/"

Si la concordancia fp32 vs int8 es menor al 98 %, avise antes de publicar (se publicaría el modelo fp32 de 111 MB).

## Publicar en Hugging Face (Space estático)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
user = api.whoami()['name']
SPACE = f'{user}/osteoclassai'
api.create_repo(SPACE, repo_type='space', space_sdk='static', exist_ok=True)
api.upload_folder(repo_id=SPACE, repo_type='space', folder_path='web',
                  ignore_patterns=['model.onnx'], commit_message='OsteoClassAI in-browser demo')
print('Listo: https://huggingface.co/spaces/' + SPACE)